# Camada Silver: base_atendentes

Trata a tabela `bronze.base_atendentes` e grava `silver.tb_base_atendentes`. Extrai o nível de atendimento, converte as datas de admissão e desligamento e valida a coerência entre status e data de desligamento.

In [0]:
from pyspark.sql import functions as F, Window

catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

## Funções auxiliares

In [0]:
def ultima_versao(df, chaves):
    """Fica com a carga mais recente de cada chave (a Bronze é append) e descarta ingestion_datetime."""
    w = Window.partitionBy(*chaves).orderBy(F.col("ingestion_datetime").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn", "ingestion_datetime")


def data_iso(c):
    """Converte texto dd/MM/yyyy em date. Vazio ou fora do padrão vira null."""
    t = F.trim(F.col(c).cast("string"))
    return F.when(t.rlike(r"^\d{2}/\d{2}/\d{4}$"), F.to_date(t, "dd/MM/yyyy"))


def checar(regra, df_falhas):
    """Para o notebook se alguma linha violar a regra."""
    n = df_falhas.count()
    assert n == 0, f"FALHA na validação '{regra}': {n} linha(s)"
    print(f"OK: {regra}")


def salvar(df, nome):
    (df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{silver_schema}.{nome}"))
    print(f"{silver_schema}.{nome}: {spark.table(f'{silver_schema}.{nome}').count()} linhas")

## 1. Atendentes

In [0]:
b_atendentes = spark.table(f"{bronze_schema}.base_atendentes")
display(b_atendentes)

In [0]:
atendentes = ultima_versao(
    b_atendentes.withColumn("id_atendente", F.trim(F.col("id_atendente")).cast("int")),
    ["id_atendente"],
)
atendentes = (
    atendentes
    .withColumn("nome_atendente", F.trim(F.col("nome_atendente")))
    # pega o dígito: '1', 'Nível 1', '2', 'Nível 2', 'N2' -> 1 ou 2
    .withColumn("nivel_atendimento", F.when(
        F.col("nivel_atendimento").rlike(r"\d"),
        F.regexp_extract(F.col("nivel_atendimento"), r"(\d)", 1).cast("int")))
    .withColumn("data_admissao", data_iso("data_admissao"))
    .withColumn("data_desligamento", data_iso("data_desligamento"))
    .withColumn("status_atendente", F.lower(F.trim(F.col("status_atendente"))))
    .select("id_atendente", "nome_atendente", "nivel_atendimento",
            "data_admissao", "data_desligamento", "status_atendente")
    .orderBy("id_atendente")
)

checar("atendentes: id, nome, nível, admissão e status preenchidos", atendentes.filter(
    F.col("id_atendente").isNull() | F.col("nome_atendente").isNull()
    | F.col("nivel_atendimento").isNull() | F.col("data_admissao").isNull()
    | F.col("status_atendente").isNull()))
checar("atendentes: id_atendente único", atendentes.groupBy("id_atendente").count().filter("count > 1"))
checar("atendentes: nível é 1 ou 2", atendentes.filter(~F.col("nivel_atendimento").isin(1, 2)))
checar("atendentes: desligamento não é anterior à admissão",
       atendentes.filter(F.col("data_desligamento") < F.col("data_admissao")))
checar("atendentes: inativo se e somente se tem data de desligamento", atendentes.filter(
    (F.col("status_atendente") == "inativo") != F.col("data_desligamento").isNotNull()))
display(atendentes)

## 2. Gravação da Silver
Só chega aqui se todas as validações passaram.

In [0]:
salvar(atendentes, "tb_base_atendentes")